In [5]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types


In [2]:
import sqlite3

def setup_mock_database():
    conn = sqlite3.connect("company.db")
    cursor = conn.cursor()
    
    # Create clean structured tables
    cursor.execute("DROP TABLE IF EXISTS orders")
    cursor.execute("DROP TABLE IF EXISTS products")
    
    cursor.execute("""
        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            name TEXT,
            price REAL
        )
    """)
    cursor.execute("""
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_email TEXT,
            product_id INTEGER,
            status TEXT,
            FOREIGN KEY(product_id) REFERENCES products(product_id)
        )
    """)
    
    # Seed real row data
    cursor.executemany("INSERT INTO products VALUES (?, ?, ?)", [
        (1, "Premium Mechanical Keyboard", 129.99),
        (2, "Ergonomic Wireless Mouse", 59.99),
        (3, "4K Ultra-Wide Monitor", 449.99)
    ])
    cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", [
        (101, "alice@email.com", 1, "Shipped"),
        (102, "bob@email.com", 3, "Processing"),
        (103, "charlie@email.com", 2, "Delivered"),
        (104, "alice@email.com", 3, "Shipped")
    ])
    
    conn.commit()
    conn.close()

setup_mock_database()
print("📁 Database created and seeded perfectly!")

📁 Database created and seeded perfectly!


Metadata

In [3]:
database_metadata = """
You are a database admin assistant. You have access to a SQL database.
Here is the exact schema of the database you are allowed to query:

Table 'products':
  - product_id (INTEGER, PRIMARY KEY)
  - name (TEXT) -> The description of the item
  - price (REAL) -> The cost of the item

Table 'orders':
  - order_id (INTEGER, PRIMARY KEY)
  - customer_email (TEXT) -> The email of the person who bought it
  - product_id (INTEGER) -> Foreign key linking to products table
  - status (TEXT) -> Can be 'Shipped', 'Processing', or 'Delivered'

Strict Rule: Only use column and table names that exist above. Do not guess.
"""

In [4]:
def run_sql_query(query_statement: str)->str:
    """Execute Read-Only query into the internal database of the company
       containing tables 'products' and 'orders' to find product and order metrics
        
       Args:
            query_statement: A valid SQLlite string (e.g SELECT * FROM order WHERE order_id=103)

          """
    try:
        conn=sqlite3.connect("company.db")
        cursor=conn.cursor()
        cursor.execute(query_statement)
        result=cursor.fetchall()
        cursor.close()
        return result
    except Exception as e:
        return f"Error happen when executing the query :{str(e)}"

In [ ]:
user_prompt="Find all the orders placed by alice@email.com and tell me what she bought."
client=genai.Client()

user_content=types.Content(
    role="user",
    parts=[
        types.Part.from_text(text=user_prompt)
    ]

)

strict_tool_calling=types.ToolConfig(
    function_calling_config=types.FunctionCallingConfig(
        mode="ANY",
        allowed_function_names=["run_sql_query"]
    )
)

model_response=client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=[user_content],
    config=types.GenerateContentConfig(
        tools=[run_sql_query],
        system_instruction=database_metadata,
        temperature=0,
        tool_config=strict_tool_calling

    )
)


if model_response.function_calls:
    tool_call=model_response.function_calls[0]
    expected_sql=tool_call.args["query_statement"]
    db_result=run_sql_query(expected_sql)

    model_turn_content=model_response.candidates[0].content
    tool_content=types.Content(
        role="tool",
        parts=[
            types.Part.from_function_response(name="run_sql_query",response={"result":db_result})
        ]
    )

    final_result=client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=[user_content,model_turn_content,tool_content],
        config=types.GenerateContentConfig(
            tools=[run_sql_query],
            temperature=0.7
        )

    )

    print(final_result.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [20]:
# ... [Your setup and first model_response call here] ...

print("--- Turn 1 Complete ---")

# Debugging check: Did Gemini even want to use the function?
if not model_response.function_calls:
    print("❌ Gemini chose NOT to use the function call.")
    print("Instead, Gemini replied with text:")
    print(model_response.text)
else:
    print("✅ Gemini successfully requested a function call!")
    tool_call = model_response.function_calls[0]
    
    # Debugging check: What arguments did Gemini actually send?
    print(f"Arguments sent by model: {tool_call.args}")
    
    # Safe dictionary check to prevent KeyError crashes
    if "query_statement" in tool_call.args:
        expected_sql = tool_call.args["query_statement"]
    elif "sql_statement" in tool_call.args:
        expected_sql = tool_call.args["sql_statement"]
    else:
        # Fallback: grab the first value in the arguments dictionary
        expected_sql = list(tool_call.args.values())[0]

    print(f"Executing SQL: {expected_sql}")
    db_result = run_sql_query(expected_sql)
    print(f"Database returned: {db_result}")

    model_turn_content = model_response.candidates[0].content
    tool_content = types.Content(
        role="tool",
        parts=[
            types.Part.from_function_response(name="run_sql_query", response={"result": db_result})
        ]
    )

    print("Sending final request to Gemini...")
    final_result = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[user_content, model_turn_content, tool_content],
        config=types.GenerateContentConfig(
            tools=[run_sql_query],
            temperature=0.7
        )
    )

    print("\n🎉 FINAL ANSWER FROM MODEL:")
    print(final_result.text)

--- Turn 1 Complete ---
❌ Gemini chose NOT to use the function call.
Instead, Gemini replied with text:
Alice ordered a "Premium Mechanical Keyboard" (order ID 101) and a "4K Ultra-Wide Monitor" (order ID 104).
